[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/normals/normals.ipynb)

# Surface Normals and Reflections

Surface normals specify either a chosen side of a surface or the winding of its vertices. Collision tests use a side; face culling uses winding. Mirroring a mesh can separate the two.

Projective geometric algebra represents both by planes. A boundary plane has extrinsic orientation, while joining ordered points gives a plane with intrinsic orientation.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from numga import NumpyContext, stack
from numga.algebras import PGA3D
from examples.util.animation import save_figure
from examples.orientation.normals import render

In [ ]:
ga = PGA3D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic

The triangle lies in a horizontal plane whose positive side is above it. Its vertices are ordered counterclockwise when viewed from above, so both normal arrows initially point upwards. A vertical mirror reflects the triangle to the other side.

In [ ]:
triangle = mv.point([[-2.4, -0.65, 0], [-0.9, -0.45, 0], [-1.8, 0.8, 0]])  # [vertices] Antivector-
side = mv.z                                                                 # [] Vector+
mirror = mv.x                                                               # [] Vector+

# Vertex order gives the joined plane its winding orientation.
winding = (triangle[0] & triangle[1] & triangle[2]).normalized()            # [] Vector-

# The same reflection acts on both planes, respecting their different orientations.
reflected_triangle = mirror >> triangle                                     # [vertices] Antivector-
reflected_side = mirror >> side                                             # [] Vector+
reflected_winding = mirror >> winding                                       # [] Vector-

vertices = stack([triangle, reflected_triangle])                            # [scenes, vertices] Antivector-
sides = stack([side, reflected_side])                                       # [scenes] Vector+
windings = stack([winding, reflected_winding])                              # [scenes] Vector-
save_figure(render.draw_normals(vertices, sides, windings, mirror), "normal_orientation");

The pale sheet is the mirror. On the left, teal arrows mark the plane's positive side, which stays above both triangles. On the right, orange edge arrows follow the vertex order; the normal arrow encodes this circulation using the handedness of the surrounding space. The mirrored winding points downward even though its triangle still has the same positive side above it. Rebuilding the winding plane from the reflected vertices agrees with reflecting the joined plane.

In matrix notation, a side covector transforms with the inverse transpose, while an area normal transforms with the cofactor matrix. A reflection separates them by a minus sign. Leo Dorst develops the distinction between intrinsic and extrinsic orientation in [Projective duality encodes complementary orientations in geometric algebras](https://doi.org/10.1002/mma.9754).

A line lying in the mirror is its own mirror image: both of its points stay where they are, so the line through them must stay too. Reflecting the joined line with `>>` leaves it as it is. The plain sandwich `mirror * line * mirror.inverse()`, which gives every element of a grade the same sign, keeps both points but reverses the line.

In [ ]:
ends = mv.point([[0, -1, 0.2], [0, 1, 0.8]])                                # [ends] Antivector-
line = ends[0] & ends[1]                                                    # [] Antibivector-

# The line, its mirror image, and the plain sandwich of the same line.
lines = stack([line, mirror >> line, mirror * line * mirror.inverse()])     # [cases] Antibivector-
save_figure(render.draw_line_in_mirror(ends, lines, mirror), "line_in_mirror");

The pale sheet is the mirror, and both points lie in it. On the left is the line, directed from the first point to the second; in the middle its mirror image, identical, since the mirror leaves it unchanged; on the right the plain sandwich, which reverses the line though neither point has moved.

The same line can be built from planes. Two planes crossing in a line that lies in the mirror, each at half a right angle to it, are each other's mirror images, so the mirror swaps them, while two points on the line stay where they are. Building from the mirror images, the join of the points is the join itself, and the meet of the swapped planes is the meet in the opposite order: by the antisymmetry of the outer product, minus the meet.

In [ ]:
crossing = stack([mv.x + mv.y, mv.y - mv.x])                                # [planes] Vector+
on_line = mv.point([[0, 0, -0.5], [0, 0, 0.5]])                             # [ends] Antivector-

# Both pairs before the mirror and after it, and the meet and the join built from each.
crossings = stack([crossing, mirror >> crossing])                           # [scenes, planes] Vector+
both_ends = stack([on_line, mirror >> on_line])                             # [scenes, ends] Antivector-
meets = crossings[:, 0] ^ crossings[:, 1]                                   # [scenes] Bivector+
joins = both_ends[:, 0] & both_ends[:, 1]                                   # [scenes] Antibivector-
save_figure(render.draw_pairs_in_mirror(crossings, both_ends, meets, joins, mirror), "pairs_in_mirror");

The pale sheet is the mirror. In the top row are the red and blue points and the line through them, in the bottom row the red and blue planes crossing in that line; on the left the scene, on the right its mirror image, in which the points have not moved and the planes have swapped places. The meet built from the mirror images is minus the meet, and the join is the join itself.